# Rivoct Voice OTP Integration Workflow

This notebook implements the Rivoct Voice OTP integration workflow, including robust error handling with exponential backoff and simulated monitoring logs.

## 1. Setup and Configuration
Import necessary libraries and define configuration constants.

In [ ]:
import requests
import time
import random
import uuid
from datetime import datetime, timedelta

# Configuration
API_URL = "https://api-wju6e5vpmq-el.a.run.app/v1/voice-otp"
API_KEY = "YOUR_API_KEY"  # Replace with your actual API key

# Mock Database
otp_sessions_db = {}

## 2. Implement Retry Strategy (Error Handling)
Define a function to handle API calls with exponential backoff for rate limits and server errors.

In [ ]:
def send_with_retry(url, payload, headers, max_retries=3):
    for attempt in range(1, max_retries + 1):
        try:
            response = requests.post(url, json=payload, headers=headers, timeout=30)
            
            if response.status_code == 200 or response.status_code == 201:
                return response.json()
            
            if response.status_code < 500 and response.status_code != 429:
                # Client error, do not retry
                print(f"Client Error ({response.status_code}): {response.text}")
                return None

            # Server error or Rate Limit
            print(f"Attempt {attempt} failed with status {response.status_code}. Retrying...")
            
        except requests.exceptions.RequestException as e:
            print(f"Attempt {attempt} failed with error: {e}")

        if attempt == max_retries:
            print("Max retries reached. Operation failed.")
            return None
        
        # Exponential backoff
        delay = (2 ** (attempt - 1))
        time.sleep(delay)
    return None

## 3. Define Send OTP Logic
Implement the logic to generate an OTP, store it, and call the API.

In [ ]:
def send_otp(phone, user_id, purpose="login"):
    # 1. Generate OTP
    otp_code = str(random.randint(100000, 999999))
    
    # 2. Store in Mock DB
    session_id = str(uuid.uuid4())
    expiry_time = datetime.now() + timedelta(minutes=10)
    
    otp_sessions_db[session_id] = {
        "phone": phone,
        "otp_code": otp_code,
        "user_id": user_id,
        "purpose": purpose,
        "expiry_time": expiry_time,
        "verified": False,
        "attempts": 0
    }
    
    print(f"[LOG] OTP Generated for {phone}: {otp_code} (Session: {session_id})")
    
    # 3. Call Rivoct API
    payload = {
        "phone": phone,
        "otpCode": otp_code,
        "metadata": {
            "userId": user_id,
            "purpose": purpose,
            "sessionId": session_id
        }
    }
    
    headers = {
        "x-api-key": API_KEY,
        "Content-Type": "application/json"
    }
    
    print(f"[LOG] Calling Rivoct API...")
    response_data = send_with_retry(API_URL, payload, headers)
    
    if response_data:
        otp_sessions_db[session_id]["rivoct_request_id"] = response_data.get("requestId")
        print(f"[LOG] API Success. Request ID: {response_data.get('requestId')}")
        return session_id
    else:
        print("[LOG] API Call Failed.")
        return None

## 4. Define Verify OTP Logic
Implement the logic to verify the OTP entered by the user.

In [ ]:
def verify_otp(session_id, input_otp):
    session = otp_sessions_db.get(session_id)
    
    if not session:
        print("[ERROR] Invalid Session ID")
        return False
        
    if session["verified"]:
        print("[ERROR] OTP Already Used")
        return False
        
    if datetime.now() > session["expiry_time"]:
        print("[ERROR] OTP Expired")
        return False
        
    if session["attempts"] >= 5:
        print("[ERROR] Too Many Attempts")
        return False
        
    if session["otp_code"] == input_otp:
        session["verified"] = True
        session["verified_at"] = datetime.now()
        print("[SUCCESS] Phone Number Verified Successfully")
        return True
    else:
        session["attempts"] += 1
        print(f"[ERROR] Invalid OTP. Attempts remaining: {5 - session['attempts']}")
        return False

## 5. Run Integration Simulation
Simulate the entire flow: Send OTP -> Receive Response -> Verify OTP.

In [ ]:
# Simulation Parameters
TEST_PHONE = "+919876543210"
TEST_USER = "user_simulation_001"

print("--- STARTING SIMULATION ---")

# Step 1: Send OTP
session_id = send_otp(TEST_PHONE, TEST_USER)

if session_id:
    # Simulate User receiving the call and entering the OTP
    # In a real scenario, this comes from the user input
    correct_otp = otp_sessions_db[session_id]["otp_code"]
    
    print(f"\n[USER ACTION] User enters OTP: {correct_otp}")
    
    # Step 2: Verify OTP
    is_verified = verify_otp(session_id, correct_otp)
    
    if is_verified:
        print("\n--- SIMULATION COMPLETED SUCCESSFULLY ---")
    else:
        print("\n--- SIMULATION FAILED AT VERIFICATION ---")
else:
    print("\n--- SIMULATION FAILED AT SENDING ---")